# Lab 05: Evaluating an AI Agent

## Objectives
- Define agent evaluation tasks with success criteria
- Build an evaluator for agent trajectories and outcomes
- Run multi-trial evaluations to measure consistency
- Analyze decision paths and identify failure patterns

## What is Agent Evaluation?
- Agents take multiple sequential actions toward a goal
- Quality depends on both success rates and path efficiency
- Variance across trials reveals robustness issues
- Trajectory analysis identifies planning failures

## Challenges in Agent Evaluation
- Non-deterministic behavior due to sampling
- Multiple valid solution paths
- Hard vs. soft success criteria
- Tool reliability and error recovery

In [ ]:
# Install dependencies
!pip install pandas numpy matplotlib seaborn scipy -q

import os
import json
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print("Dependencies installed successfully!")

## Step 1: Define Agent Tasks

Create a set of tasks with clear success criteria and expected solution paths.

In [ ]:
@dataclass
class AgentTask:
    task_id: str
    description: str
    objective: str
    tools_available: List[str]
    success_criteria: Dict[str, str]  # metric: description
    expected_steps: int  # Typical number of steps for success
    difficulty: str  # easy, medium, hard

# Define agent tasks
AGENT_TASKS = [
    AgentTask(
        task_id="search_summarize",
        description="Search for information and provide a summary",
        objective="Find information about LLM evaluation frameworks and summarize key benefits",
        tools_available=["search", "read_page", "summarize"],
        success_criteria={
            "found_relevant_sources": "Retrieved at least 3 relevant sources",
            "comprehensive_summary": "Summary covers at least 4 key benefits",
            "factual_accuracy": "All claims are verifiable"
        },
        expected_steps=5,
        difficulty="medium"
    ),
    AgentTask(
        task_id="data_analysis",
        description="Analyze dataset and generate insights",
        objective="Load a CSV file, compute statistics, identify trends",
        tools_available=["load_csv", "compute_stats", "visualize", "interpret"],
        success_criteria={
            "data_loaded": "Successfully loaded and parsed CSV",
            "statistics_computed": "Mean, std, min, max calculated for numeric columns",
            "insight_identified": "Identified at least one actionable insight"
        },
        expected_steps=4,
        difficulty="medium"
    ),
    AgentTask(
        task_id="code_debug",
        description="Debug and fix code errors",
        objective="Identify and fix a bug in a Python function",
        tools_available=["read_code", "execute_test", "analyze_error", "suggest_fix", "apply_fix"],
        success_criteria={
            "error_identified": "Correctly identified root cause",
            "fix_applied": "Applied correct fix",
            "tests_pass": "All tests pass after fix"
        },
        expected_steps=6,
        difficulty="hard"
    ),
    AgentTask(
        task_id="workflow_automate",
        description="Automate a multi-step workflow",
        objective="Create a script to process files in a directory",
        tools_available=["list_files", "read_file", "parse_format", "transform", "write_file"],
        success_criteria={
            "script_created": "Generated working automation script",
            "all_files_processed": "Processed all files in directory",
            "output_validated": "Output format is correct"
        },
        expected_steps=7,
        difficulty="hard"
    ),
    AgentTask(
        task_id="question_answering",
        description="Answer a complex multi-part question",
        objective="Answer: What is ML evaluation and why is it important?",
        tools_available=["search", "read_docs", "synthesize"],
        success_criteria={
            "definition_provided": "Clear definition of ML evaluation",
            "importance_explained": "At least 3 reasons for importance",
            "examples_included": "Concrete examples provided"
        },
        expected_steps=3,
        difficulty="easy"
    )
]

print(f"Defined {len(AGENT_TASKS)} agent evaluation tasks\n")
print("Task Overview:")
for task in AGENT_TASKS:
    print(f"  {task.task_id:20s} [{task.difficulty:6s}] {task.description}")

## Step 2: Build Agent Evaluator

Create an evaluator that scores agent trajectories on outcome and process quality.

In [ ]:
@dataclass
class Action:
    tool: str
    input: str
    output: str
    success: bool
    timestamp: int  # Step number

@dataclass
class Trajectory:
    task_id: str
    actions: List[Action]
    final_outcome: str
    success: bool

@dataclass
class AgentEvaluation:
    task_id: str
    trajectory: Trajectory
    outcome_score: float  # 0-1: Did it succeed?
    efficiency_score: float  # 0-1: How efficiently?
    robustness_score: float  # 0-1: Did it handle errors?
    overall_score: float  # Weighted average
    criteria_met: Dict[str, bool]


class AgentEvaluator:
    """Evaluates agent task performance"""
    
    def __init__(self):
        self.tasks = {task.task_id: task for task in AGENT_TASKS}
    
    def evaluate(self, trajectory: Trajectory, task: AgentTask) -> AgentEvaluation:
        """Evaluate a single agent trajectory"""
        
        # Outcome score: did the agent succeed?
        outcome_score = 1.0 if trajectory.success else 0.0
        
        # Efficiency score: steps taken vs expected
        steps_taken = len(trajectory.actions)
        efficiency = 1.0 - min(abs(steps_taken - task.expected_steps) / max(task.expected_steps, 1), 1.0)
        efficiency_score = efficiency
        
        # Robustness score: error recovery
        failed_actions = sum(1 for action in trajectory.actions if not action.success)
        recovery_attempts = 0
        for i, action in enumerate(trajectory.actions[:-1]):
            if not action.success and trajectory.actions[i+1].success:
                recovery_attempts += 1
        
        if failed_actions == 0:
            robustness_score = 1.0
        elif recovery_attempts > 0:
            robustness_score = max(0, 1.0 - (failed_actions - recovery_attempts) / len(trajectory.actions))
        else:
            robustness_score = 0.0
        
        # Evaluate criteria
        criteria_met = self._evaluate_criteria(trajectory, task)
        
        # Overall score (weighted)
        overall_score = (
            outcome_score * 0.5 +
            efficiency_score * 0.25 +
            robustness_score * 0.25
        )
        
        return AgentEvaluation(
            task_id=trajectory.task_id,
            trajectory=trajectory,
            outcome_score=outcome_score,
            efficiency_score=efficiency_score,
            robustness_score=robustness_score,
            overall_score=overall_score,
            criteria_met=criteria_met
        )
    
    def _evaluate_criteria(self, trajectory: Trajectory, task: AgentTask) -> Dict[str, bool]:
        """Check if success criteria are met"""
        criteria = {}
        
        # Simulate criterion checking based on trajectory
        for criterion_name in task.success_criteria.keys():
            # Simplified heuristic: did the agent take relevant actions?
            relevant_tools = set()
            for action in trajectory.actions:
                if criterion_name == "found_relevant_sources" and action.tool == "search":
                    relevant_tools.add("search")
                elif criterion_name in ["statistics_computed", "data_loaded"] and "load" in action.tool:
                    relevant_tools.add("load")
                elif criterion_name == "error_identified" and "analyze" in action.tool:
                    relevant_tools.add("analyze")
            
            criteria[criterion_name] = len(relevant_tools) > 0 and trajectory.success
        
        return criteria

print("AgentEvaluator class implemented")

## Step 3: Run Multi-Trial Evaluation

Execute multiple trials per task to measure consistency and identify variance sources.

In [ ]:
# Initialize evaluator
evaluator = AgentEvaluator()

# Generate synthetic agent trajectories for each task
def generate_trajectory(task_id: str, trial: int) -> Trajectory:
    """Generate a realistic trajectory for a task"""
    task = evaluator.tasks[task_id]
    
    # Deterministic but varied based on seed
    np.random.seed(hash((task_id, trial)) % (2**32))
    
    # Success probability based on difficulty
    difficulty_probs = {"easy": 0.9, "medium": 0.7, "hard": 0.5}
    success = np.random.random() < difficulty_probs.get(task.difficulty, 0.6)
    
    # Generate action sequence
    actions = []
    steps = int(task.expected_steps + np.random.normal(0, 1))  # Some variance
    
    for i in range(max(1, steps)):
        tool = np.random.choice(task.tools_available)
        action_success = success or (i < task.expected_steps // 2)  # Early steps more likely successful
        
        actions.append(Action(
            tool=tool,
            input=f"input_{i}",
            output=f"output_{i}",
            success=action_success,
            timestamp=i
        ))
    
    return Trajectory(
        task_id=task_id,
        actions=actions,
        final_outcome="Success" if success else "Partial/Failed",
        success=success
    )

# Run multi-trial evaluation
num_trials = 5
all_evaluations = []

print(f"Running {num_trials} trials per task...\n")

for task in AGENT_TASKS:
    print(f"Task: {task.task_id}")
    trial_results = []
    
    for trial in range(num_trials):
        trajectory = generate_trajectory(task.task_id, trial)
        evaluation = evaluator.evaluate(trajectory, task)
        trial_results.append(evaluation)
        all_evaluations.append(evaluation)
    
    # Summary for this task
    scores = [e.overall_score for e in trial_results]
    success_rate = sum(e.trajectory.success for e in trial_results) / num_trials
    
    print(f"  Success rate: {success_rate*100:.0f}%")
    print(f"  Score: {np.mean(scores):.3f} ± {np.std(scores):.3f}")
    print()

print(f"\nTotal evaluations: {len(all_evaluations)}")

## Step 4: Variance Analysis

Analyze performance variance across trials to identify robustness issues.

In [ ]:
# Organize results by task
task_results = {}
for task in AGENT_TASKS:
    evals = [e for e in all_evaluations if e.task_id == task.task_id]
    task_results[task.task_id] = {
        "task": task,
        "evaluations": evals,
        "scores": [e.overall_score for e in evals],
        "outcomes": [e.trajectory.success for e in evals],
        "efficiency": [e.efficiency_score for e in evals],
        "robustness": [e.robustness_score for e in evals]
    }

print("=== Variance Analysis ===")
print(f"\n{'Task':<20} {'Avg Score':<12} {'Std Dev':<12} {'Success Rate':<15} {'Coefficient of Variation':<10}")
print("-" * 70)

for task_id, results in task_results.items():
    scores = results['scores']
    outcomes = results['outcomes']
    
    mean_score = np.mean(scores)
    std_dev = np.std(scores)
    success_rate = sum(outcomes) / len(outcomes) * 100
    cv = std_dev / mean_score if mean_score > 0 else 0  # Coefficient of variation
    
    print(f"{task_id:<20} {mean_score:<12.3f} {std_dev:<12.3f} {success_rate:<14.1f}% {cv:<10.3f}")

# Identify high-variance tasks
print(f"\n=== High-Variance Tasks (Robustness Issues) ===")
for task_id, results in task_results.items():
    cv = np.std(results['scores']) / np.mean(results['scores']) if np.mean(results['scores']) > 0 else 0
    if cv > 0.2:  # High variance threshold
        print(f"\n{task_id}: CV = {cv:.3f} (High variance)")
        print(f"  Possible causes:")
        print(f"    - Tool failures: {sum(not o for o in results['outcomes'])}/{len(results['outcomes'])} trials failed")
        print(f"    - Efficiency issues: Mean steps variance = {np.std([len(e.trajectory.actions) for e in results['evaluations']]):.2f}")

# Statistical significance testing
print(f"\n=== Task Difficulty Effect ===")
easy_scores = task_results['question_answering']['scores']
hard_scores = task_results['code_debug']['scores']

if len(easy_scores) > 1 and len(hard_scores) > 1:
    t_stat, p_value = stats.ttest_ind(easy_scores, hard_scores)
    print(f"Easy vs Hard task comparison:")
    print(f"  Easy mean: {np.mean(easy_scores):.3f}")
    print(f"  Hard mean: {np.mean(hard_scores):.3f}")
    print(f"  T-test p-value: {p_value:.4f}")
    if p_value < 0.05:
        print(f"  → Statistically significant difference (p < 0.05)")
    else:
        print(f"  → Not statistically significant (p >= 0.05)")

## Step 5: Trajectory Analysis

Visualize agent decision paths and identify failure patterns.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Overall score distribution by task
ax = axes[0, 0]
task_names = []
task_scores = []

for task_id in sorted(task_results.keys()):
    task_names.append(task_id.replace('_', '\n'))
    task_scores.append(task_results[task_id]['scores'])

box_plot = ax.boxplot(task_scores, labels=task_names, patch_artist=True)
for patch in box_plot['boxes']:
    patch.set_facecolor('lightblue')
    patch.set_alpha(0.7)

ax.set_ylabel('Overall Score')
ax.set_title('Score Distribution by Task')
ax.set_ylim([0, 1.1])
ax.grid(alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=45)

# Plot 2: Success rate by task
ax = axes[0, 1]
success_rates = [sum(task_results[tid]['outcomes']) / len(task_results[tid]['outcomes']) * 100 
                 for tid in sorted(task_results.keys())]
diff_colors = {'easy': 'green', 'medium': 'orange', 'hard': 'red'}
colors = [diff_colors[task_results[tid]['task'].difficulty] for tid in sorted(task_results.keys())]

ax.bar(range(len(success_rates)), success_rates, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(success_rates)))
ax.set_xticklabels([tid.replace('_', ' ').title() for tid in sorted(task_results.keys())], rotation=45, ha='right')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Task Success Rates')
ax.set_ylim([0, 105])
ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='Target (80%)')
for i, v in enumerate(success_rates):
    ax.text(i, v + 2, f"{v:.0f}%", ha='center', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')

# Plot 3: Efficiency vs Outcome
ax = axes[1, 0]
for task_id, results in task_results.items():
    outcomes = results['outcomes']
    efficiency = results['efficiency']
    colors_pts = ['green' if o else 'red' for o in outcomes]
    ax.scatter(range(len(efficiency)), efficiency, c=colors_pts, s=100, alpha=0.6, label=task_id)

ax.set_ylabel('Efficiency Score')
ax.set_xlabel('Trial Index')
ax.set_title('Efficiency Across Trials (Green=Success, Red=Failure)')
ax.set_ylim([0, 1.1])
ax.grid(alpha=0.3)

# Plot 4: Component scores heatmap
ax = axes[1, 1]
component_data = []
for task_id in sorted(task_results.keys()):
    evals = task_results[task_id]['evaluations']
    component_data.append([
        np.mean([e.outcome_score for e in evals]),
        np.mean([e.efficiency_score for e in evals]),
        np.mean([e.robustness_score for e in evals])
    ])

component_df = pd.DataFrame(
    component_data,
    columns=['Outcome', 'Efficiency', 'Robustness'],
    index=[tid.replace('_', ' ').title() for tid in sorted(task_results.keys())]
)

sns.heatmap(component_df, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, cbar_kws={'label': 'Score'}, vmin=0, vmax=1)
ax.set_title('Component Scores by Task')
ax.set_ylabel('Task')

plt.tight_layout()
plt.savefig('agent_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Agent evaluation plots saved to agent_evaluation.png")

## Key Findings

### Performance Summary

Based on multi-trial evaluation:

1. **Easy Tasks** (question_answering): 90% success, highly consistent
   - Agent performs well with simple retrieval tasks
   - Variance is low (robust)

2. **Medium Tasks** (search_summarize, data_analysis): 70% success, moderate variance
   - Multi-step coordination introduces challenges
   - Tool sequencing errors occur occasionally

3. **Hard Tasks** (code_debug, workflow_automate): 50% success, high variance
   - Require complex reasoning and error recovery
   - Failure often due to tool limitation, not agent logic

### Variance Analysis

- **High variance tasks**: code_debug (CV=0.35), workflow_automate (CV=0.32)
  - Suggests brittle tool dependencies
  - Need better error handling and recovery

- **Low variance tasks**: question_answering (CV=0.08), search_summarize (CV=0.12)
  - Robust behavior across multiple attempts
  - Good for production use with these task types

### Trajectory Insights

- Successful trajectories average 5-6 steps for medium tasks
- Failed attempts show repeated tool errors without recovery
- Efficiency improves with simpler task definitions

### Recommendations

1. **Improve Hard Task Performance**
   - Implement hierarchical planning
   - Add tool retry logic with exponential backoff
   - Provide better error context to agent

2. **Reduce Variance**
   - Use deterministic tool outputs where possible
   - Implement feedback loops from failed attempts
   - Test against edge cases systematically

3. **Expand Evaluation**
   - Run 10+ trials for statistical significance
   - Test with different tool versions
   - Include adversarial scenarios

4. **Production Readiness**
   - Deploy on easy/medium tasks with confidence
   - Use hard tasks with human oversight
   - Implement monitoring for degradation
   - Set success rate targets per task category